# Debug

In [4]:
# --- CELL 1: Setup ---
import os
import sys
from pathlib import Path
from datetime import datetime

# ============================================================================
# BASE PATH — All data saves to this folder
# ============================================================================
BASE_PATH = r"C:\Users\Sahil\Women-safety meraki"
os.makedirs(BASE_PATH, exist_ok=True)

# ============================================================================
# PROJECT STRUCTURE — Define all paths relative to BASE_PATH
# ============================================================================
RAW_DIR       = os.path.join(BASE_PATH, 'data', 'raw', 'chicago')
PROCESSED_DIR = os.path.join(BASE_PATH, 'data', 'processed')
MODELS_DIR    = os.path.join(BASE_PATH, 'data', 'models')
DOCS_DIR      = os.path.join(BASE_PATH, 'docs')
FIGURES_DIR   = os.path.join(DOCS_DIR, 'figures')
TABLES_DIR    = os.path.join(DOCS_DIR, 'tables')

for folder in [RAW_DIR, PROCESSED_DIR, MODELS_DIR, FIGURES_DIR, TABLES_DIR]:
    os.makedirs(folder, exist_ok=True)

# ============================================================================
# CONFIG CONSTANTS (inlined — replaces src.config import)
# ============================================================================
H3_RESOLUTION = 9

# (min_lon, min_lat, max_lon, max_lat) — WGS84
BBOX_CHICAGO = (-87.95, 41.64, -87.52, 42.03)
BBOX_DELHI   = (76.84, 28.40, 77.35, 28.88)

# ============================================================================
# IMPORTS
# ============================================================================
import pandas as pd
import numpy as np
import requests
import time
import h3
import json

# ============================================================================
# H3 UTILITY FUNCTIONS (inlined — replaces src.h3_utils import)
# ============================================================================
def vectorized_latlng_to_cell(lats, lngs, resolution):
    """Vectorized H3 assignment — 10x faster than apply()."""
    return [
        h3.latlng_to_cell(lat, lng, resolution)
        for lat, lng in zip(lats, lngs)
    ]

def get_neighbors(cell, k=1):
    """
    Return the target cell plus all neighbors up to k rings.
    k=1 → 7 cells (self + 6 neighbors)
    k=2 → 19 cells (self + 6 + 12)
    """
    return list(h3.grid_disk(cell, k))

# ============================================================================
# CITY SELECTION
# ============================================================================
CITY = "chicago"  # Change to "delhi" for Indian data
BBOX = BBOX_CHICAGO if CITY == "chicago" else BBOX_DELHI

# Overpass API mirrors (rotating helps avoid throttling)
OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://z.overpass-api.de/api/interpreter",
]

print(f"✅ Ready | City: {CITY}")
print(f"📁 Processed data → {PROCESSED_DIR}")
print(f"📁 Figures        → {FIGURES_DIR}")

✅ Ready | City: chicago
📁 Processed data → C:\Users\Sahil\Women-safety meraki\data\processed
📁 Figures        → C:\Users\Sahil\Women-safety meraki\docs\figures


In [5]:
# --- CELL 2: Robust Overpass Query Function ---

def query_overpass(query: str, description: str = "", max_retries: int = 3):
    """
    Query Overpass API with proper headers, mirror rotation, retry logic.
    
    FIXES:
    ✅ User-Agent header (Overpass returns 406 without it)
    ✅ Mirror rotation (if one is down, try next)
    ✅ Exponential backoff on 429 rate limits
    """
    headers = {
        'User-Agent': 'Meraki-Research/1.0 (BSc Data Science Thesis Project)',
        'Accept': 'application/json'
    }
    
    for mirror in OVERPASS_MIRRORS:
        for attempt in range(max_retries):
            try:
                print(f"⏳ {description} | Mirror: {mirror.split('/')[2]} | Attempt {attempt+1}")
                
                response = requests.post(
                    mirror,
                    data={'data': query},
                    headers=headers,
                    timeout=180
                )
                
                if response.status_code == 429:
                    wait = 30 * (attempt + 1)
                    print(f"   ⚠️ Rate limited. Waiting {wait}s...")
                    time.sleep(wait)
                    continue
                
                if response.status_code == 406:
                    print(f"   ⚠️ Mirror rejected request, trying next...")
                    break
                
                response.raise_for_status()
                data = response.json()
                
                # Parse elements
                records = []
                for el in data.get('elements', []):
                    lat = el.get('lat') or el.get('center', {}).get('lat')
                    lon = el.get('lon') or el.get('center', {}).get('lon')
                    if lat and lon:
                        records.append({'lat': lat, 'lon': lon})
                
                df = pd.DataFrame(records)
                print(f"   ✅ Retrieved {len(df)} features")
                return df
                
            except requests.exceptions.Timeout:
                print(f"   ⚠️ Timeout, retrying...")
                time.sleep(10)
            except Exception as e:
                print(f"   ⚠️ Error: {e}")
                time.sleep(5)
        
        time.sleep(3)  # Between mirrors
    
    print(f"   ❌ All mirrors failed for '{description}'")
    return pd.DataFrame()


# ============================================================================
# SAFE BBOX FORMATTER
# Overpass expects: south,west,north,east
# Our BBOX_CHICAGO = (min_lon, min_lat, max_lon, max_lat)  → (west, south, east, north)
# So the correct mapping is: south=BBOX[1], west=BBOX[0], north=BBOX[3], east=BBOX[2]
# ============================================================================
if isinstance(BBOX, dict):
    bbox_str = f"{BBOX['south']},{BBOX['west']},{BBOX['north']},{BBOX['east']}"
elif isinstance(BBOX, (tuple, list)) and len(BBOX) == 4:
    # Our convention: (min_lon, min_lat, max_lon, max_lat)
    west, south, east, north = BBOX
    bbox_str = f"{south},{west},{north},{east}"
elif isinstance(BBOX, str):
    bbox_str = BBOX
else:
    # Default Chicago fallback (south,west,north,east)
    bbox_str = "41.6445,-87.9401,42.0230,-87.5240"

print(f"🗺️  Bounding box string: {bbox_str}")

🗺️  Bounding box string: 41.64,-87.95,42.03,-87.52


In [6]:
# --- CELL 3: Query All POI Categories ---

poi_queries = {
    'bus_stops': f"""
        [out:json][timeout:180];
        (
          node["highway"="bus_stop"]({bbox_str});
          node["public_transport"="stop_position"]["bus"="yes"]({bbox_str});
        );
        out center;
    """,
    'transit_stations': f"""
        [out:json][timeout:180];
        (
          node["railway"="station"]({bbox_str});
          node["railway"="halt"]({bbox_str});
          node["station"="subway"]({bbox_str});
        );
        out center;
    """,
    'bars_liquor': f"""
        [out:json][timeout:180];
        (
          node["amenity"="bar"]({bbox_str});
          node["amenity"="pub"]({bbox_str});
          node["amenity"="nightclub"]({bbox_str});
          node["shop"="alcohol"]({bbox_str});
        );
        out center;
    """,
    'police_stations': f"""
        [out:json][timeout:180];
        (
          node["amenity"="police"]({bbox_str});
          way["amenity"="police"]({bbox_str});
        );
        out center;
    """,
    'street_lights': f"""
        [out:json][timeout:240];
        (
          node["highway"="street_lamp"]({bbox_str});
        );
        out;
    """,
    'parks': f"""
        [out:json][timeout:180];
        (
          way["leisure"="park"]({bbox_str});
        );
        out center;
    """,
}

poi_data = {}
for poi_type, query in poi_queries.items():
    df = query_overpass(query, description=poi_type)
    poi_data[poi_type] = df
    time.sleep(8)  # ✅ FIX: Rate limit courtesy between queries

# Summary
print("\n📊 POI Collection Results:")
for name, df in poi_data.items():
    print(f"   {name:20s}: {len(df):>6,} features")

⏳ bus_stops | Mirror: overpass-api.de | Attempt 1
   ✅ Retrieved 19334 features
⏳ transit_stations | Mirror: overpass-api.de | Attempt 1
   ⚠️ Rate limited. Waiting 30s...
⏳ transit_stations | Mirror: overpass-api.de | Attempt 2
   ✅ Retrieved 271 features
⏳ bars_liquor | Mirror: overpass-api.de | Attempt 1
   ✅ Retrieved 1036 features
⏳ police_stations | Mirror: overpass-api.de | Attempt 1
   ✅ Retrieved 104 features
⏳ street_lights | Mirror: overpass-api.de | Attempt 1
   ✅ Retrieved 4575 features
⏳ parks | Mirror: overpass-api.de | Attempt 1
   ✅ Retrieved 1554 features

📊 POI Collection Results:
   bus_stops           : 19,334 features
   transit_stations    :    271 features
   bars_liquor         :  1,036 features
   police_stations     :    104 features
   street_lights       :  4,575 features
   parks               :  1,554 features


In [8]:
# --- CELL 4: Fallback to Synthetic if any query returned zero ---

def generate_synthetic_pois(city: str = "chicago"):
    """Fallback synthetic POI data if Overpass fails"""
    np.random.seed(42)
    center_lat = BBOX['center_lat']
    center_lng = BBOX['center_lng']
    spread = 0.15
    
    configs = {
        'bus_stops': 2000,
        'transit_stations': 150,
        'bars_liquor': 800,
        'police_stations': 50,
        'street_lights': 5000,
        'parks': 300
    }
    
    synthetic = {}
    for name, count in configs.items():
        lats = np.random.normal(center_lat, spread, count)
        lngs = np.random.normal(center_lng, spread, count)
        synthetic[name] = pd.DataFrame({'lat': lats, 'lon': lngs})
    
    return synthetic

# Use synthetic for any empty results
for name, df in poi_data.items():
    if len(df) == 0:
        print(f"⚠️  {name} returned no data, using synthetic fallback")
        poi_data[name] = generate_synthetic_pois()[name]

In [9]:
# --- CELL 5: Aggregate POIs to H3 Cells (with neighborhood expansion) ---

def assign_pois_to_cells_with_neighbors(df_poi: pd.DataFrame, 
                                         poi_name: str, 
                                         k_neighbors: int = 1) -> pd.DataFrame:
    """
    Count POIs in each H3 cell PLUS its k-ring neighbors.
    
    ✅ FIX: Original code only counted POIs in exact cell.
    A bus stop 50m away (adjacent cell) matters for pedestrian safety.
    """
    if len(df_poi) == 0:
        return pd.DataFrame(columns=['h3_cell', poi_name])
    
    # Assign each POI to its H3 cell
    df_poi = df_poi.copy()
    df_poi['h3_cell'] = vectorized_latlng_to_cell(
        df_poi['lat'].values, df_poi['lon'].values, H3_RESOLUTION
    )
    
    # Count per cell
    direct_counts = df_poi.groupby('h3_cell').size().to_dict()
    all_cells = set(direct_counts.keys())
    
    # Expand: for each cell, sum its own + neighbors' POI counts
    expanded_counts = {}
    for cell in all_cells:
        try:
            neighbors = get_neighbors(cell, k=k_neighbors)
            expanded_counts[cell] = sum(direct_counts.get(n, 0) for n in neighbors)
        except Exception:
            expanded_counts[cell] = direct_counts.get(cell, 0)
    
    return pd.DataFrame(
        list(expanded_counts.items()),
        columns=['h3_cell', poi_name]
    )

# Aggregate all POIs
poi_features = None
for poi_name, df in poi_data.items():
    cell_counts = assign_pois_to_cells_with_neighbors(df, poi_name, k_neighbors=1)
    
    if poi_features is None:
        poi_features = cell_counts
    else:
        poi_features = poi_features.merge(cell_counts, on='h3_cell', how='outer')

poi_features = poi_features.fillna(0)

# Convert to int (POI counts must be integers)
for col in poi_features.columns:
    if col != 'h3_cell':
        poi_features[col] = poi_features[col].astype(int)

print(f"✅ POI features for {len(poi_features):,} cells")
print(poi_features.describe().round(1))

✅ POI features for 5,574 cells
       bus_stops  transit_stations  bars_liquor  police_stations  \
count     5574.0            5574.0       5574.0           5574.0   
mean        16.5               0.1          0.7              0.0   
std         15.1               0.5          3.1              0.1   
min          0.0               0.0          0.0              0.0   
25%          4.0               0.0          0.0              0.0   
50%         14.0               0.0          0.0              0.0   
75%         25.0               0.0          0.0              0.0   
max        104.0              11.0         55.0              2.0   

       street_lights   parks  
count         5574.0  5574.0  
mean             3.3     0.6  
std             16.5     1.5  
min              0.0     0.0  
25%              0.0     0.0  
50%              0.0     0.0  
75%              0.0     0.0  
max            409.0    14.0  


In [10]:
from pathlib import Path

BASE_PATH = Path(r"C:\Users\Sahil\Women-safety meraki")

RAW_DIR       = BASE_PATH / 'data' / 'raw' / 'chicago'
PROCESSED_DIR = BASE_PATH / 'data' / 'processed'
MODELS_DIR    = BASE_PATH / 'data' / 'models'
DOCS_DIR      = BASE_PATH / 'docs'
FIGURES_DIR   = DOCS_DIR / 'figures'
TABLES_DIR    = DOCS_DIR / 'tables'

for folder in [RAW_DIR, PROCESSED_DIR, MODELS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [11]:
# --- CELL 6: Save ---
output_path = PROCESSED_DIR / f"{CITY}_poi_features.parquet"
poi_features.to_parquet(output_path, index=False)

print(f"💾 Saved: {output_path.name}")
print(f"   Full path: {output_path}")
print(f"📊 Feature summary:")
for col in poi_features.columns:
    if col != 'h3_cell':
        vals = poi_features[col]
        print(f"   {col:20s}: mean={vals.mean():.1f}, "
              f"max={vals.max()}, non-zero cells={(vals > 0).sum()}")

print(f"\n✅ Notebook 03 complete!")

💾 Saved: chicago_poi_features.parquet
   Full path: C:\Users\Sahil\Women-safety meraki\data\processed\chicago_poi_features.parquet
📊 Feature summary:
   bus_stops           : mean=16.5, max=104, non-zero cells=4337
   transit_stations    : mean=0.1, max=11, non-zero cells=258
   bars_liquor         : mean=0.7, max=55, non-zero cells=602
   police_stations     : mean=0.0, max=2, non-zero cells=103
   street_lights       : mean=3.3, max=409, non-zero cells=733
   parks               : mean=0.6, max=14, non-zero cells=1372

✅ Notebook 03 complete!


In [ ]:
# --- CELL 6: Save ---

output_path = PROCESSED_DIR / f"{CITY}_poi_features.parquet"
poi_features.to_parquet(output_path, index=False)

print(f"💾 Saved: {output_path.name}")
print(f"📊 Feature summary:")
for col in poi_features.columns:
    if col != 'h3_cell':
        vals = poi_features[col]
        print(f"   {col:20s}: mean={vals.mean():.1f}, "
              f"max={vals.max()}, non-zero cells={(vals > 0).sum()}")

print(f"\n✅ Notebook 03 complete!")